In [1]:
import pandas as pd
import numpy as np
import textstat
import string
import spacy
import re

import seaborn as sns
import matplotlib.pyplot as plt

import pyphen
from wordfreq import zipf_frequency

import math
from collections import Counter
import networkx as nx

import nltk
nltk.download('cmudict')
textstat.set_lang("pt")
from nltk.corpus import wordnet as wn

[nltk_data] Downloading package cmudict to
[nltk_data]     C:\Users\luan.barbosa\AppData\Roaming\nltk_data...
[nltk_data]   Package cmudict is already up-to-date!


## (1) Functions and Processing

In [11]:
# Surface Features

def n_words(doc):
    words = [token for token in doc if not token.is_space]
    return max(len(words), 1)

def n_sentences(doc):
    sentences = list(doc.sents)
    return max(len(sentences), 1)

def words_per_sentence(doc):
    return n_words(doc) / n_sentences(doc)

def n_characters(doc):
    return len(doc.text.replace(" ", ""))

def reading_time(doc, wpm=260):
    words = n_words(doc)
    return words / wpm

def punctuation_features(doc):
    text = doc.text
    punct_set = set(string.punctuation)

    total_punct = sum(1 for ch in text if ch in punct_set)
    n_w = n_words(doc)
    n_s = n_sentences(doc)

    exclam = text.count("!")
    quest = text.count("?")

    return {
        "total_punct": total_punct,
        "punct_per_word": total_punct / n_w,
        "punct_per_sentence": total_punct / n_s,
        "exclam": exclam,
        "quest": quest
    }

def uppercase_ratio(doc):
    text = doc.text
    letters = [ch for ch in text if ch.isalpha()]
    if not letters:
        return 0.0
    
    upper = sum(1 for ch in letters if ch.isupper())
    return upper / len(letters)

def char_repetition_ratio(doc):
    pattern = re.compile(r"(.)\1{2,}") # 3+

    words = doc.text.split()
    if not words:
        return 0.0
    
    repeated = sum(1 for w in words if pattern.search(w.lower()))
    return repeated / len(words)

def avg_letters_per_word(doc):
    text = doc.text
    words = re.findall(r'\b\w+\b', text, flags=re.UNICODE)
    if not words:
        return 0.0

    total_letters = sum(len(w) for w in words)
    return total_letters / len(words)

dic = pyphen.Pyphen(lang='pt_BR')
def avg_syllables_per_word(doc):
    text = doc.text
    words = re.findall(r'\b\w+\b', text, flags=re.UNICODE)
    if not words:
        return 0.0

    syllables = sum(len(dic.inserted(w).split('-')) for w in words)
    return syllables / len(words)

In [3]:
# Lexical Features

def lexical_frequency(doc):
    freqs = []

    for token in doc:
        if token.is_alpha:
            freq = zipf_frequency(token.text.lower(), 'pt')
            freqs.append(freq)

    if not freqs:
        return {
            "rarest_word": 0.0,
            "most_common_word": 0.0,
            "avg_freq": 0.0,
            "freq_disp": 0.0
        }
    
    return {
        "rarest_word": min(freqs),
        "most_common_word": max(freqs),
        "avg_freq": np.mean(freqs),
        "freq_disp": np.std(freqs)
    }

def ttr(doc):
    tokens = [t.text.lower() for t in doc if t.is_alpha]

    if not tokens:
        return 0.0
    
    unique = set(tokens)
    return len(unique) / len(tokens)

def stopword_ratio(doc):
    words = [t for t in doc if t.is_alpha]
    if not words:
        return 0.0

    stopwords = [t for t in words if t.is_stop]
    
    return len(stopwords) / len(words)

def n_unique_words(doc):
    tokens = [t.text.lower() for t in doc if t.is_alpha]

    return len(set(tokens))

def mean_unique_word_length(doc):
    tokens = [t.text.lower() for t in doc if t.is_alpha]

    unique = set(tokens)
    if not unique:
        return 0.0

    return np.mean([len(w) for w in unique])

def count_syllables(word):
    return len(dic.inserted(word).split('-'))

def flesch_portuguese(doc):
    sentences = list(doc.sents)
    words = [t.text for t in doc if t.is_alpha]

    if not sentences or not words:
        return 0.0

    n_sent = len(sentences)
    n_words = len(words)
    n_syll = sum(count_syllables(w) for w in words)

    flesch = 226 - 1.04 * (n_words / n_sent) - 72 * (n_syll / n_words)
    return flesch

In [4]:
# Syntactic Features

POS_TAGS = [
    "NOUN", "VERB", "ADJ", "ADV", "PRON", "DET",
    "ADP", "AUX", "CCONJ", "SCONJ", "NUM", "PROPN", "INTJ"
]

def pos_tag_counts(doc):
    counts = {pos: 0 for pos in POS_TAGS}

    for token in doc:
        if token.pos_ in counts:
            counts[token.pos_] += 1
    
    return counts

def mean_verbs_per_sentence(doc):
    sentences = list(doc.sents)
    if len(sentences) == 0:
        return 0.0

    verb_counts = []

    for sent in sentences:
        verbs = sum(1 for tok in sent if tok.pos_ == "VERB")
        verb_counts.append(verbs)

    return np.mean(verb_counts)

def mean_aux_per_sentence(doc):
    sentences = list(doc.sents)
    if len(sentences) == 0:
        return 0.0

    aux_counts = []

    for sent in sentences:
        aux = sum(1 for tok in sent if tok.pos_ == "AUX")
        aux_counts.append(aux)

    return np.mean(aux_counts)

def total_auxiliaries(doc):
    return sum(1 for tok in doc if tok.pos_ == "AUX")

def tree_depth(token):
    if not list(token.children):
        return 1
    return 1 + max(tree_depth(child) for child in token.children)

def mean_dependency_tree_depth(doc):
    depths = []

    for sent in doc.sents:
        root = sent.root
        depths.append(tree_depth(root))

    if len(depths) == 0:
        return 0.0

    return np.mean(depths)

def dependency_relation_counts(doc):
    dep_counts = {}

    for token in doc:
        dep = token.dep_
        dep_counts[dep] = dep_counts.get(dep, 0) + 1

    return dep_counts

def mean_dependency_length(doc):
    distances = []

    for token in doc:
        if token.head != token:
            distance = abs(token.i - token.head.i)
            distances.append(distance)

    if len(distances) == 0:
        return 0.0

    return np.mean(distances)

SUBORDINATE_DEPS = {"advcl", "ccomp", "xcomp", "acl", "relcl"}

def mean_subordinate_clauses(doc):
    sentences = list(doc.sents)
    if len(sentences) == 0:
        return 0.0

    counts = []

    for sent in sentences:
        sub_count = sum(
            1 for tok in sent
            if tok.dep_ in SUBORDINATE_DEPS
        )
        counts.append(sub_count)

    return np.mean(counts)

In [5]:
# Other Features (Discourse and Semantics)

def lexical_ambiguity(doc):
    ambiguities = []

    for token in doc:
        if token.is_alpha:
            synsets = wn.synsets(token.text, lang='por')
            ambiguities.append(len(synsets))

    if (len(ambiguities) == 0):
        return 0.0

    return sum(ambiguities) / len(ambiguities)

def lexical_entropy(doc):
    tokens = [t.lemma_.lower() for t in doc if t.is_alpha]

    if len(tokens) == 0:
        return 0.0
    
    counts = Counter(tokens)
    total = len(tokens)

    entropy = 0.0
    for count in counts.values():
        p = count / total
        entropy -= p * math.log(p)
    
    return entropy

CONNECTIVES_PT = {
    "porque", "pois", "mas", "porém", "entretanto",
    "logo", "portanto", "assim", "além disso",
    "embora", "se", "quando", "enquanto"
}

def cohesion_ratio(doc):
    tokens = [t for t in doc if t.is_alpha]
    if len(tokens) == 0:
        return 0.0

    count = sum(
        1 for t in tokens
        if t.text.lower() in CONNECTIVES_PT
    )

    return count / len(tokens)

DISCOURSE_MARKERS = {
    "então", "assim", "portanto", "porém",
    "logo", "ou seja", "bem", "agora"
}

def discourse_markers_ratio(doc):
    tokens = [t for t in doc if t.is_alpha]
    if len(tokens) == 0:
        return 0.0
    
    count = sum(
        1 for token in doc
        if token.text.lower() in DISCOURSE_MARKERS
    )

    return count / len(tokens)

def children_dependency_tree(doc):
    children_counts = [
        len(list(token.children))
        for token in doc
    ]

    if len(children_counts) == 0:
        return 0.0

    return {
        "mean_children_per_token": sum(children_counts) / len(children_counts),
        "max_children": max((len(list(token.children)) for token in doc), default=0),
        "min_children": min((len(list(token.children)) for token in doc), default=0),
    }
    
def root_verb_position(doc):
    positions = []

    for sent in doc.sents:
        tokens = list(sent)
        if len(tokens) == 0:
            continue

        root = sent.root

        pos = (root.i - sent.start) / len(tokens)
        positions.append(pos)

    if len(positions) == 0:
        return 0.0
    
    return np.mean(positions)

def dependency_pattern_diversity(doc):
    patterns = set(
        (t.head.pos_, t.dep_, t.pos_)
        for t in doc
    )

    return len(patterns)

def subtree_size(doc):
    sizes = [
        len(list(token.subtree))
        for token in doc
    ]

    if len(sizes) == 0:
        return 0.0

    return {
        "mean_subtree_size": np.mean(sizes),
        "max_subtree_size": max((len(list(token.subtree)) for token in doc), default=0),
        "min_subtree_size": min((len(list(token.subtree)) for token in doc), default=0),
    }

def named_entity_ratio(doc):
    tokens = [t for t in doc if t.is_alpha]
    if len(tokens) == 0:
        return 0.0

    return len(doc.ents) / len(tokens)

def entity_type_diversity(doc):
    return len(set(ent.label_ for ent in doc.ents))

def dependency_graph_metrics(doc):
    G = nx.Graph()

    for token in doc:
        G.add_node(token.i)
        if token.head != token:
            G.add_edge(token.i, token.head.i)

    if len(G.nodes) == 0:
        return {
            "avg_degree": 0.0,
            "density": 0.0,
            "clustering": 0.0
        }

    avg_degree = np.mean([d for _, d in G.degree()])
    density = nx.density(G)
    clustering = nx.average_clustering(G)

    return {
        "avg_degree": avg_degree,
        "density": density,
        "clustering": clustering
    }

def passive_voice_ratio(doc):
    sentences = list(doc.sents)

    if len(sentences) == 0:
        return 0.0

    passive_count = 0

    for sent in sentences:
        if any("pass" in token.dep_ for token in sent):
            passive_count += 1

    return passive_count / len(sentences)

## (2) Features for GoEmotions

In [8]:
df = pd.read_csv("../data/treated/go_emotions_treated.csv")

df['UNCLEAN_TEXT_PT'] = (
    df['UNCLEAN_TEXT_PT']
    .fillna("")        # remove NaN
    .astype(str)       # garante string
)

print(df.shape)
df.head(3)

(54234, 35)


,id,admiration,amusement,anger,annoyance,approval,caring,confusion,curiosity,desire,...,remorse,sadness,surprise,neutral,text,texto,UNCLEAN_TEXT_PT,BASE_TEXT_PT,UNCLEAN_TEXT_EN,BASE_TEXT_EN
0,eczazk6,0,0,0,0,1,0,0,0,0,...,0,0,0,0,Fast as [NAME] will carry me. Seriously uptown...,Tão rápido quanto [NOME] me carregará. Seriame...,Tão rápido quanto [NOME] me carregará. Seriame...,tao rapido quanto nome carregara seriamente up...,Fast as [NAME] will carry me. Seriously uptown...,fast name will carry seriously uptown downtown...
1,eczb07q,0,0,0,0,0,0,0,0,0,...,0,0,0,1,You blew it. They played you like a fiddle.,Você estragou isso. Eles tocaram você como um ...,Você estragou isso. Eles tocaram você como um ...,voce estragou isso eles tocaram voce como violino,You blew it. They played you like a fiddle.,you blew they played you like fiddle
2,eczb4bm,0,0,0,0,0,0,0,0,0,...,0,0,0,0,TL;DR No more Superbowls for [NAME]. Get ready...,TL;DR Não há mais Super Bowls para [NAME]. Pre...,TL;DR Não há mais Super Bowls para [NAME]. Pre...,nao mais super bowls para name prepare-se para...,TL;DR No more Superbowls for [NAME]. Get ready...,more superbowls for name get ready for another...


In [12]:
text_col = 'UNCLEAN_TEXT_PT'

nlp = spacy.load("pt_core_news_lg")
docs = list(nlp.pipe(df[text_col], batch_size=1000))
df['doc'] = docs

In [13]:
print("-----------")
print("Surface Features")
# ---- Surface Features ----
df['n_words'] = df['doc'].apply(n_words)
df['n_sentences'] = df['doc'].apply(n_sentences)
df['words_per_sentence'] = df['doc'].apply(words_per_sentence)
df['n_characters'] = df['doc'].apply(n_characters)
df['reading_time'] = df['doc'].apply(reading_time)
df['uppercase_ratio'] = df['doc'].apply(uppercase_ratio)
df['char_repetition_ratio'] = df['doc'].apply(char_repetition_ratio)
df['avg_letters_per_word'] = df['doc'].apply(avg_letters_per_word)
df['avg_syllables_per_word'] = df['doc'].apply(avg_syllables_per_word)
# punctuation
punct_df = df['doc'].apply(punctuation_features).apply(pd.Series)
df = pd.concat([df, punct_df], axis=1)

print("-----------")
print("Lexical Features")
# ---- Lexical Features ----
# lexical_frequency ----
lexfreq_df = df['doc'].apply(lexical_frequency).apply(pd.Series)
df = pd.concat([df, lexfreq_df], axis=1)
df['ttr'] = df['doc'].apply(ttr)
df['stopword_ratio'] = df['doc'].apply(stopword_ratio)
df['n_unique_words'] = df['doc'].apply(n_unique_words)
df['mean_unique_word_length'] = df['doc'].apply(mean_unique_word_length)
df['flesch_portuguese'] = df['doc'].apply(flesch_portuguese)

print("-----------")
print("Syntactic Features")
# ---- Syntactic ----
pos_df = df['doc'].apply(pos_tag_counts).apply(pd.Series)
df = pd.concat([df, pos_df], axis=1)
df['mean_verbs_per_sentence'] = df['doc'].apply(mean_verbs_per_sentence)
df['mean_aux_per_sentence'] = df['doc'].apply(mean_aux_per_sentence)
df['total_auxiliaries'] = df['doc'].apply(total_auxiliaries)
df['mean_dependency_tree_depth'] = df['doc'].apply(mean_dependency_tree_depth)
dep_df = df['doc'].apply(dependency_relation_counts).apply(pd.Series)
df = pd.concat([df, dep_df], axis=1)
df['mean_dependency_length'] = df['doc'].apply(mean_dependency_length)
df['mean_subordinate_clauses'] = df['doc'].apply(mean_subordinate_clauses)

print("-----------")
print("Semantics Features")
# ---- Discourse / Semantics ----
df['lexical_ambiguity'] = df['doc'].apply(lexical_ambiguity)
df['lexical_entropy'] = df['doc'].apply(lexical_entropy)
df['cohesion_ratio'] = df['doc'].apply(cohesion_ratio)
df['discourse_markers_ratio'] = df['doc'].apply(discourse_markers_ratio)
children_df = df['doc'].apply(children_dependency_tree).apply(pd.Series)
df = pd.concat([df, children_df], axis=1)
df['root_verb_position'] = df['doc'].apply(root_verb_position)
df['dependency_pattern_diversity'] = df['doc'].apply(dependency_pattern_diversity)
subtree_df = df['doc'].apply(subtree_size).apply(pd.Series)
df = pd.concat([df, subtree_df], axis=1)
df['named_entity_ratio'] = df['doc'].apply(named_entity_ratio)
df['entity_type_diversity'] = df['doc'].apply(entity_type_diversity)
graph_df = df['doc'].apply(dependency_graph_metrics).apply(pd.Series)
df = pd.concat([df, graph_df], axis=1)
df['passive_voice_ratio'] = df['doc'].apply(passive_voice_ratio)

-----------
Surface Features
-----------
Lexical Features
-----------
Syntactic Features
-----------
Semantics Features


In [14]:
emotion_cols = [
    'admiration', 'amusement', 'anger', 'annoyance',
    'approval', 'caring', 'confusion', 'curiosity', 'desire',
    'disappointment', 'disapproval', 'disgust', 'embarrassment',
    'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love',
    'nervousness', 'optimism', 'pride', 'realization', 'relief',
    'remorse', 'sadness', 'surprise', 'neutral'
]

base_cols = ["UNCLEAN_TEXT_PT", "BASE_TEXT_PT"]
remove_cols = ['TEXT_NO_STOP', 'CLEAN_TEXT', 'TEXT_LEMMA', 'texto']

feature_cols = [c for c in df.columns if c not in base_cols and c not in remove_cols and c not in emotion_cols]

df_final = df[base_cols + emotion_cols + feature_cols]

df_final.columns

Index(['UNCLEAN_TEXT_PT', 'BASE_TEXT_PT', 'admiration', 'amusement', 'anger',
       'annoyance', 'approval', 'caring', 'confusion', 'curiosity',
       ...
       'dependency_pattern_diversity', 'mean_subtree_size', 'max_subtree_size',
       'min_subtree_size', 'named_entity_ratio', 'entity_type_diversity',
       'avg_degree', 'density', 'clustering', 'passive_voice_ratio'],
      dtype='object', length=131)

In [15]:
df_final.to_csv(
    "../data/feat/go_emotions_with_features.csv",
    index=False
)